# Train causal sentence classifiers (fastText + SVM) trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/causal_detection/fasttext_classifier.py`,
`src/causal_detection/embedding_classifier.py`, `scripts/causal_classification/`), đồng bộ
qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/dataset/causal_sentences.csv` (đã kiểm nghiệm/verify từ ensemble 3
   model, cột `label` = `causal`/`non_causal`) vào
   `DRIVE_ROOT/data/dataset/causal_sentences.csv` trên Drive.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [ ]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và model train xong được giữ lại qua các session, không cần
copy tay mỗi lần mở lại Colab.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/dataset/ 2>/dev/null || echo "Chưa có data/dataset/causal_sentences.csv trên Drive — upload trước khi train."


## 4. Cài thư viện

In [ ]:
!pip install -q -r requirements.txt


## 5. Train fastText classifier

Đọc `data/dataset/causal_sentences.csv`, train/test split 80/20 (stratify theo `label`),
in `classification_report`, lưu model vào `models/causal_classifier_fasttext.bin`
(= Drive, qua symlink ở bước 3).

In [ ]:
!python -m scripts.causal_classification.train_fasttext_classifier


## 6. Train SVM classifier (sentence embedding + SVM)

Cùng data, encode câu bằng `keepitreal/vietnamese-sbert` rồi train `SVC`. Lưu model vào
`models/causal_classifier.joblib`. Bước encode embedding chạy nhanh hơn nhiều nếu có GPU
(Runtime > Change runtime type > GPU).

In [ ]:
!python -m scripts.causal_classification.train_causal_classifier


## 7. (Tuỳ chọn) Đăng nhập Hugging Face Hub

Cần nếu muốn push 2 model đã train lên Hub ở bước cuối. Token tạo tại
https://huggingface.co/settings/tokens (quyền write).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 8. (Tuỳ chọn) Push model lên Hugging Face Hub

Lưu bản train xong lên Hub để có version history riêng cho model, tách khỏi git, và tải
lại được từ máy local hoặc lần train sau mà không cần train lại. Mỗi classifier push lên
một repo riêng (fastText và SVM/joblib không dùng chung format với nhau). Repo ID lấy từ
`CAUSAL_FASTTEXT_HF_REPO_ID` / `CAUSAL_CLASSIFIER_HF_REPO_ID` trong `configs/config.py` —
đổi ở đó nếu muốn push sang repo khác, script train local và notebook này dùng chung 1 nguồn.


In [ ]:
from configs.config import CAUSAL_CLASSIFIER_HF_REPO_ID, CAUSAL_FASTTEXT_HF_REPO_ID
from src.causal_detection.fasttext_classifier import FastTextCausalClassifier
from src.causal_detection.embedding_classifier import EmbeddingCausalClassifier

fasttext_classifier = FastTextCausalClassifier.load()
fasttext_classifier.push_to_hub(CAUSAL_FASTTEXT_HF_REPO_ID)

svm_classifier = EmbeddingCausalClassifier.load()
svm_classifier.push_to_hub(CAUSAL_CLASSIFIER_HF_REPO_ID)
